<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_ML_cell_to_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Pure ML Cell-Wise Telecom Power Prediction

## Google Colab Python Code

# ============================================================
# PURE MACHINE LEARNING APPROACH
# CELL-WISE POWER PREDICTION
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)

# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]

# ============================================================
# FLAT 2G + 3G POWER
# ============================================================

site_db['power_2g'] = site_db['RRU_2G'] * 150

site_db['power_3g'] = site_db['RRU_3G'] * 200

site_db['flat_2g_3g_power'] = (

    site_db['power_2g'] +
    site_db['power_3g']

)

# ============================================================
# PREPARE 4G CELL DATA
# ============================================================

lte_df = traffic_4g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

# ============================================================
# PREPARE 5G CELL DATA
# ============================================================

nr_df = traffic_5g.merge(
    site_db,
    on='Site_ID',
    how='left'
)

# ============================================================
# CELL COUNTS
# ============================================================

lte_counts = (
    lte_df.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()
)

nr_counts = (
    nr_df.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()
)

lte_df['lte_cell_count'] = (
    lte_df['Site_ID'].map(lte_counts)
)

nr_df['nr_cell_count'] = (
    nr_df['Site_ID'].map(nr_counts)
)

# ============================================================
# AGGREGATE CELL TRAFFIC
# ============================================================

lte_site_traffic = (

    lte_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

lte_site_traffic.rename(
    columns={'traffic_load_mbps': 'total_4g_traffic'},
    inplace=True
)

nr_site_traffic = (

    nr_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

nr_site_traffic.rename(
    columns={'traffic_load_mbps': 'total_5g_traffic'},
    inplace=True
)

# ============================================================
# PREPARE LTE TRAINING DATA
# ============================================================

lte_train = lte_df.merge(

    lte_site_traffic,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

lte_train = lte_train.merge(

    nr_site_traffic,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

lte_train = lte_train.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

lte_train.fillna(0, inplace=True)

# ============================================================
# ESTIMATED LTE CELL TARGET
# ============================================================

lte_train['lte_cell_target_power'] = (

    (
        lte_train['site_power'] -
        lte_train['flat_2g_3g_power']
    )

    *

    (
        lte_train['traffic_load_mbps']
        /
        (
            lte_train['total_4g_traffic'] +
            lte_train['total_5g_traffic'] +
            1
        )
    )

)

# ============================================================
# LTE FEATURES
# ============================================================

lte_features = [

    'traffic_load_mbps',
    'lte_cell_count',
    'RRU_4G',
    'Boards_4G',
    'BBU3900',
    'BBU3910',
    'trigger_ID'

]

X_lte = lte_train[lte_features]

y_lte = lte_train['lte_cell_target_power']

# ============================================================
# LTE TRAIN TEST SPLIT
# ============================================================

X_train_lte, X_test_lte, y_train_lte, y_test_lte = train_test_split(

    X_lte,
    y_lte,
    test_size=0.2,
    random_state=42

)

# ============================================================
# LTE RANDOM FOREST MODEL
# ============================================================

lte_model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

lte_model.fit(X_train_lte, y_train_lte)

# ============================================================
# LTE CELL POWER PREDICTION
# ============================================================

lte_df['predicted_lte_cell_power'] = (

    lte_model.predict(
        lte_df[lte_features]
    )

)

# ============================================================
# PREPARE NR TRAINING DATA
# ============================================================

nr_train = nr_df.merge(

    lte_site_traffic,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

nr_train = nr_train.merge(

    nr_site_traffic,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

nr_train = nr_train.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

nr_train.fillna(0, inplace=True)

# ============================================================
# ESTIMATED NR CELL TARGET
# ============================================================

nr_train['nr_cell_target_power'] = (

    (
        nr_train['site_power'] -
        nr_train['flat_2g_3g_power']
    )

    *

    (
        nr_train['traffic_load_mbps']
        /
        (
            nr_train['total_4g_traffic'] +
            nr_train['total_5g_traffic'] +
            1
        )
    )

)

# ============================================================
# NR FEATURES
# ============================================================

nr_features = [

    'traffic_load_mbps',
    'nr_cell_count',
    'AAU_5G',
    'Boards_5G',
    'BBU5900',
    'trigger_ID'

]

X_nr = nr_train[nr_features]

y_nr = nr_train['nr_cell_target_power']

# ============================================================
# NR TRAIN TEST SPLIT
# ============================================================

X_train_nr, X_test_nr, y_train_nr, y_test_nr = train_test_split(

    X_nr,
    y_nr,
    test_size=0.2,
    random_state=42

)

# ============================================================
# NR RANDOM FOREST MODEL
# ============================================================

nr_model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

nr_model.fit(X_train_nr, y_train_nr)

# ============================================================
# NR CELL POWER PREDICTION
# ============================================================

nr_df['predicted_nr_cell_power'] = (

    nr_model.predict(
        nr_df[nr_features]
    )

)

# ============================================================
# LTE SITE POWER AGGREGATION
# ============================================================

lte_site_prediction = (

    lte_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['predicted_lte_cell_power']

    .sum()

)

# ============================================================
# NR SITE POWER AGGREGATION
# ============================================================

nr_site_prediction = (

    nr_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['predicted_nr_cell_power']

    .sum()

)

# ============================================================
# MERGE LTE + NR PREDICTIONS
# ============================================================

final_df = lte_site_prediction.merge(

    nr_site_prediction,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='outer'

)

final_df.fillna(0, inplace=True)

# ============================================================
# ADD 2G + 3G FLAT POWER
# ============================================================

final_df = final_df.merge(

    site_db[
        ['Site_ID', 'flat_2g_3g_power']
    ],

    on='Site_ID',
    how='left'

)

# ============================================================
# FINAL SITE POWER PREDICTION
# ============================================================

final_df['final_predicted_power'] = (

    final_df['predicted_lte_cell_power'] +
    final_df['predicted_nr_cell_power'] +
    final_df['flat_2g_3g_power']

)

# ============================================================
# ADD ACTUAL SITE POWER
# ============================================================

final_df = final_df.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

# ============================================================
# MODEL EVALUATION
# ============================================================

mae = mean_absolute_error(

    final_df['site_power'],
    final_df['final_predicted_power']

)

rmse = np.sqrt(

    mean_squared_error(

        final_df['site_power'],
        final_df['final_predicted_power']

    )

)

mape = np.mean(

    np.abs(

        (
            final_df['site_power'] -
            final_df['final_predicted_power']
        )

        /

        final_df['site_power']

    )

) * 100

r2 = r2_score(

    final_df['site_power'],
    final_df['final_predicted_power']

)

# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('PURE ML MODEL PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

# ============================================================
# ERROR CALCULATION
# ============================================================

final_df['error'] = (

    final_df['site_power'] -
    final_df['final_predicted_power']

)

final_df['error_percentage'] = (

    np.abs(final_df['error'])
    /
    final_df['site_power']

) * 100

# ============================================================
# EXPORT RESULTS
# ============================================================

final_df.to_excel(

    'Pure_ML_Site_Power_Predictions.xlsx',
    index=False

)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Pure_ML_Site_Power_Predictions.xlsx')

# ============================================================
# SAMPLE RESULTS
# ============================================================

print(final_df.head(20))
print("Done")


PURE ML MODEL PERFORMANCE
MAE  : 207.03
RMSE : 315.83
MAPE : 5.11 %
R2   : 0.9765
OUTPUT FILE CREATED
Pure_ML_Site_Power_Predictions.xlsx
    Site_ID  trigger_ID        date          datetime  \
0       101           1  2026-03-01  2026-03-01 00:00   
1       101           1  2026-03-02  2026-03-02 00:00   
2       101           1  2026-03-03  2026-03-03 00:00   
3       101           1  2026-03-04  2026-03-04 00:00   
4       101           1  2026-03-05  2026-03-05 00:00   
5       101           1  2026-03-06  2026-03-06 00:00   
6       101           1  2026-03-07  2026-03-07 00:00   
7       101           2  2026-03-01  2026-03-01 00:15   
8       101           2  2026-03-02  2026-03-02 00:15   
9       101           2  2026-03-03  2026-03-03 00:15   
10      101           2  2026-03-04  2026-03-04 00:15   
11      101           2  2026-03-05  2026-03-05 00:15   
12      101           2  2026-03-06  2026-03-06 00:15   
13      101           2  2026-03-07  2026-03-07 00:15   
14     